# Faz 3 — YOLOv8 Baseline Eğitimi ve Değerlendirme (Fold 0)

**Amaç:** Faz 2'de hazırlanan YOLO veri seti (fold 0) üzerinde Ultralytics YOLOv8 baseline modelini eğitmek ve makaleye uyumlu **F1 @ IoU top-5 ortalama** metriğiyle değerlendirmek.

**Notebook adımları**
1. Ortam + GPU kontrolü
2. Eğitim (kısa hızlı test: epoch=10; tam çalışma: epoch=50)
3. En iyi ağırlıkla val seti üzerinde çıkarım
4. F1 @ IoU ∈ {0.1, 0.2, 0.3, 0.4, 0.5} → top-5 ortalama F1
5. Sınıf bazında precision/recall/F1 + confusion özet
6. (Opsiyonel) hasta-düzeyi 3D post-processing demo

**Süre tahmini:**
- T4/P100 GPU + 10 epoch + ~10K annotasyonlu kesit: ~30-60 dk
- Tam 50 epoch: 3-5 saat (Kaggle 30sa/hafta limiti içinde kalır)

## 1. Ortam

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = DATA_ROOT / "Eğitim Verisi" / "Eg╠åitim Verisi.zip"

os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
os.environ["ABDOMEN_OUT_DIR"]      = str(CODE / "outputs")

sys.path.insert(0, str(CODE))

from src.config import DET_DATA_DIR, CKPT_DIR, SUPER_CLASSES, DEFAULT_DET

fold = 0
fold_dir = DET_DATA_DIR / f"fold{fold}"
dataset_yaml = fold_dir / "dataset.yaml"

print("YOLO veri seti:", fold_dir, "→ var?", fold_dir.exists())
print("dataset.yaml :", dataset_yaml.exists())
print("Checkpoint   :", CKPT_DIR)

In [ ]:
# GPU/Donanım kontrolü
import torch
print(f"PyTorch         : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
elif torch.backends.mps.is_available():
    print("MPS device      : Apple Silicon")
else:
    print("⚠️  GPU yok — eğitim CPU'da çok yavaş olacaktır. Kaggle/Colab GPU runtime önerilir.")

## 2. Eğitim Ayarları

`src/config.py` → `DEFAULT_DET` varsayılanlarını kullanıyoruz; ilk hızlı test için epoch=10 ve daha küçük model (yolov8s) öneririz.

In [ ]:
from dataclasses import replace

# Hızlı baseline ayarı (1 fold prove-of-concept)
fast_cfg = replace(DEFAULT_DET,
    model="yolov8s.pt",   # 11M parametre — T4'te bile rahat sığar
    epochs=10,            # kısa hızlı test; tam çalışma için 50 yapın
    img_size=512,
    batch_size=16,
    patience=5,
)
print("Eğitim konfigürasyonu:")
for k, v in fast_cfg.__dict__.items():
    print(f"  {k}: {v}")

## 3. Eğitim

In [ ]:
from src.detection import train_yolo

# Bu çağrı:
#  - Ultralytics YOLO ile dataset.yaml üzerinde eğitir
#  - En iyi ağırlığı project/exp/weights/best.pt altına yazar
#  - Eğitim sonu best.pt'yi outputs/checkpoints/yolo_fold0.pt'ye kopyalar (detection.py'de tanımlı)
weights_path = train_yolo(fold=fold, cfg=fast_cfg, project=str(CODE / "runs" / "det"))
print("\nEn iyi ağırlık:", weights_path)

## 4. Validation Setinde Çıkarım

In [ ]:
# Ultralytics direkt val seti üzerinde değerlendirmeyi de yapar.
# Burada manuel olarak val PNG'lerinde inference yapıp `predictions.csv` üretiyoruz —
# böylece kendi F1@IoU değerlendirmemizi koşturabiliriz.

import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO
from tqdm import tqdm

model = YOLO(str(weights_path))
val_dir = fold_dir / "images" / "val"
val_imgs = sorted(val_dir.glob("*.png"))
print(f"Val görüntü sayısı: {len(val_imgs)}")

CONF_TH = 0.05   # düşük tut; F1@IoU eğrisi sonradan threshold'u tarayacak
predictions = []
for img_path in tqdm(val_imgs, desc="val inference"):
    # PNG adı: "{case}_{image_id}.png" formatında (export_yolo_dataset'in çıktısı)
    stem = img_path.stem
    if "_" in stem:
        case, image_id = stem.split("_", 1)
    else:
        case, image_id = stem, "0"
    res = model.predict(str(img_path), conf=CONF_TH, verbose=False, imgsz=fast_cfg.img_size)[0]
    for box, score, cls in zip(res.boxes.xyxy.cpu().numpy(),
                               res.boxes.conf.cpu().numpy(),
                               res.boxes.cls.cpu().numpy()):
        predictions.append({
            "case": int(case),
            "image_id": int(image_id),
            "class": int(cls),
            "score": float(score),
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
        })

pred_df = pd.DataFrame(predictions)
pred_out = CODE / "outputs" / "logs" / f"yolo_fold{fold}_val_preds.csv"
pred_out.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(pred_out, index=False)
print(f"Tahmin kayıt: {len(pred_df):,} → {pred_out}")

## 5. Ground Truth Oluştur (val PNG'leri için)

In [ ]:
# YOLO label dosyalarından (normalize cx,cy,w,h) GT'yi xyxy piksel formatına çeviriyoruz.
import re

gt_rows = []
val_lbl_dir = fold_dir / "labels" / "val"
for lbl_path in val_lbl_dir.glob("*.txt"):
    img_path = val_dir / (lbl_path.stem + ".png")
    if not img_path.exists():
        continue
    W, H = Image.open(img_path).size
    stem = lbl_path.stem
    if "_" in stem:
        case, image_id = stem.split("_", 1)
    else:
        case, image_id = stem, "0"
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        cls = int(parts[0])
        cx, cy, w, h = [float(x) for x in parts[1:5]]
        x1 = (cx - w/2) * W; y1 = (cy - h/2) * H
        x2 = (cx + w/2) * W; y2 = (cy + h/2) * H
        gt_rows.append({
            "case": int(case), "image_id": int(image_id),
            "class": cls,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        })

gt_df = pd.DataFrame(gt_rows)
print(f"GT BBox sayısı: {len(gt_df):,}")
print(gt_df.head(3))

## 6. F1 @ IoU Değerlendirmesi (Top-5 ortalama)

In [ ]:
from src.evaluation import f1_at_iou, top5_f1_mean

# Top-5 ortalama F1 (makale standardı)
result = top5_f1_mean(pred_df, gt_df)
print("Top-5 ortalama F1 :", round(result['top5_mean_f1'], 4))
print("Top-5 eşikleri    :", [(round(t,2), round(f,4)) for t,f in result['top5']])
print("\nTüm eşiklerde:")
for t, f in result['per_threshold']:
    print(f"  IoU={t:.1f} → macro F1={f:.4f}")

In [ ]:
# Sınıf bazında özet (IoU=0.3'te)
detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
print(f"Macro F1 @ IoU 0.3: {detail['macro_f1']:.4f}")
print(f"Micro F1 @ IoU 0.3: {detail['micro_f1']:.4f}\n")

rows = []
for cls_id, m in detail['per_class'].items():
    rows.append({
        "class_id": cls_id,
        "class_name": SUPER_CLASSES[cls_id] if cls_id < len(SUPER_CLASSES) else f"id{cls_id}",
        "precision": round(m['precision'], 3),
        "recall":    round(m['recall'], 3),
        "f1":        round(m['f1'], 3),
        "TP": m['tp'], "FP": m['fp'], "FN": m['fn'],
    })
cls_table = pd.DataFrame(rows).sort_values("class_id")
cls_table

## 7. Görsel Sağlama — 4 Rastgele Val Tahmini

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

random.seed(11)
samples = random.sample(val_imgs, min(4, len(val_imgs)))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img_path in zip(axes, samples):
    img = np.array(Image.open(img_path))
    ax.imshow(img)
    stem = img_path.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    case, image_id = int(case), int(image_id)
    # GT (mavi)
    g = gt_df[(gt_df['case'] == case) & (gt_df['image_id'] == image_id)]
    for _, r in g.iterrows():
        ax.add_patch(mpatches.Rectangle((r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
                                        linewidth=2, edgecolor='blue', facecolor='none', linestyle='--'))
    # Pred (kırmızı)
    p = pred_df[(pred_df['case'] == case) & (pred_df['image_id'] == image_id) & (pred_df['score'] >= 0.25)]
    for _, r in p.iterrows():
        ax.add_patch(mpatches.Rectangle((r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
                                        linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(r.x1, r.y1-3, f"{SUPER_CLASSES[r['class']][:12]} ({r.score:.2f})",
                color='red', fontsize=8, backgroundcolor='white')
    ax.set_title(f"{case}/{image_id}", fontsize=9)
    ax.axis('off')
plt.suptitle("Mavi=GT, Kırmızı=Tahmin (conf≥0.25)", fontsize=11)
plt.tight_layout(); plt.show()

## 8. Sonuç Çıktısı

In [ ]:
import json

result_path = CODE / "outputs" / "logs" / f"yolo_fold{fold}_metrics.json"
with open(result_path, "w") as f:
    json.dump({
        "fold": fold,
        "config": fast_cfg.__dict__,
        "top5_mean_f1": result['top5_mean_f1'],
        "per_threshold": [(round(t,2), round(f,5)) for t,f in result['per_threshold']],
        "per_class_at_iou_0.3": {SUPER_CLASSES[cid]: {k: round(v,4) if isinstance(v,float) else v
                                                       for k,v in m.items()}
                                 for cid, m in detail['per_class'].items()},
    }, f, indent=2, ensure_ascii=False)

print("Metrik raporu:", result_path)
print(f"\n📊 ÖZET — Fold 0 YOLO Baseline:")
print(f"   Top-5 ortalama F1: {result['top5_mean_f1']:.4f}")
print(f"   Macro F1 @ IoU 0.3: {detail['macro_f1']:.4f}")
print(f"   Micro F1 @ IoU 0.3: {detail['micro_f1']:.4f}")

## 9. Faz 3 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| Eğitilmiş ağırlık | `outputs/checkpoints/yolo_fold0.pt` |
| Val tahminleri | `outputs/logs/yolo_fold0_val_preds.csv` |
| Metrik raporu | `outputs/logs/yolo_fold0_metrics.json` |

**Sonraki adımlar (sonraki sürümler için):**
- Tam 50 epoch ile yeniden eğitim (Kaggle P100/T4×2)
- 5-fold çapraz doğrulama (1→5)
- Hasta-düzeyi 3D post-processing (`detection.predict_volume` ile)
- nnDetection 3B karşılaştırması (Faz 4)
- Yarışma seti üzerinde harici test